# Sesión 41 · Visión, audio y multimodal

Aplica el flujo procesador → modelo → post-proceso a modalidades distintas al texto.

**Objetivos:**
1. Clasificar imágenes (ViT) y detectar objetos (DETR) con bounding boxes.
2. Segmentar imágenes semánticamente y visualizar máscaras.
3. Clasificar imágenes sin ejemplos previos con CLIP (zero-shot).
4. Transcribir audio con Whisper.
5. Describir imágenes con un modelo multimodal (image-to-text).

> **Formato:** ~30 min explicación · ~75 min notebook · ~15 min reto.

In [ ]:
%pip install -q transformers timm Pillow requests

## Utilidad: cargar imágenes desde URL o ruta local

Usaremos esta función en todos los ejemplos. Acepta tanto URLs como rutas locales.

In [ ]:
from PIL import Image
import requests
from io import BytesIO

def cargar_imagen(origen):
    """Carga una imagen desde una URL http(s) o desde una ruta local."""
    if str(origen).startswith("http"):
        respuesta = requests.get(origen, stream=True,
                                 headers={"User-Agent": "Mozilla/5.0"})
        respuesta.raise_for_status()
        return Image.open(BytesIO(respuesta.content)).convert("RGB")
    return Image.open(origen).convert("RGB")

# Imagen de prueba: coches en una calle (Wikimedia Commons, dominio público)
URL_COCHES = (
    "https://upload.wikimedia.org/wikipedia/commons/thumb/"
    "b/b9/Good_Smile_Company_offices_ladies.jpg/"
    "800px-Good_Smile_Company_offices_ladies.jpg"
)

imagen = cargar_imagen(URL_COCHES)
display(imagen)

---
## 1 · Clasificación de imágenes (ViT)

El modelo `google/vit-base-patch16-224` divide la imagen en parches de 16×16 px
y los trata como tokens (igual que BERT con palabras).

In [ ]:
from transformers import pipeline

clasificador_img = pipeline(
    "image-classification",
    model="google/vit-base-patch16-224"
)

URL_GATO = (
    "https://upload.wikimedia.org/wikipedia/commons/thumb/"
    "4/4d/Cat_November_2010-1a.jpg/320px-Cat_November_2010-1a.jpg"
)

imagen_gato = cargar_imagen(URL_GATO)
display(imagen_gato)

resultados = clasificador_img(imagen_gato)
for r in resultados[:5]:
    print(f"{r['label']:40s}  {r['score']:.3f}")

---
## 2 · Detección de objetos con DETR + bounding boxes

DETR (*Detection Transformer*, Facebook/Meta) aplica atención directamente sobre
la imagen y predice cajas delimitadoras sin necesidad de propuestas de regiones.

Usamos **dos formas** para ver la diferencia entre la API de bajo nivel y el pipeline:

### 2a · API de bajo nivel (procesador + modelo)

In [ ]:
import torch
from PIL import ImageDraw
from transformers import DetrImageProcessor, DetrForObjectDetection

procesador = DetrImageProcessor.from_pretrained("facebook/detr-resnet-50")
modelo_detr = DetrForObjectDetection.from_pretrained("facebook/detr-resnet-50")
modelo_detr.eval()

imagen_det = cargar_imagen(URL_COCHES)

entradas = procesador(images=imagen_det, return_tensors="pt")

with torch.no_grad():
    salidas = modelo_detr(**entradas)

target_sizes = torch.tensor([imagen_det.size[::-1]])
resultados_det = procesador.post_process_object_detection(
    salidas, target_sizes=target_sizes, threshold=0.9
)[0]

# Dibujar bounding boxes
lienzo = imagen_det.copy()
draw = ImageDraw.Draw(lienzo)

for score, label, box in zip(
    resultados_det["scores"],
    resultados_det["labels"],
    resultados_det["boxes"]
):
    box_px = [round(v, 1) for v in box.tolist()]
    etiqueta = modelo_detr.config.id2label[label.item()]
    confianza = score.item() * 100
    print(f"{etiqueta:20s}  {confianza:.1f}%  {box_px}")
    draw.rectangle(box_px, outline="green", width=3)
    draw.text((box_px[0], box_px[1] - 12), etiqueta, fill="green")

display(lienzo)

### 2b · Pipeline de alto nivel (más sencillo)

In [ ]:
detector = pipeline("object-detection", model="facebook/detr-resnet-50")

detecciones = detector(imagen_det)

lienzo2 = imagen_det.copy()
draw2 = ImageDraw.Draw(lienzo2)

for obj in detecciones:
    box = list(obj["box"].values())  # xmin, ymin, xmax, ymax
    draw2.rectangle(box, outline="blue", width=3)
    draw2.text(
        (box[0], box[1] - 12),
        f"{obj['label']} {obj['score']:.2f}",
        fill="blue"
    )

display(lienzo2)

---
## 3 · Segmentación semántica y visualización de máscaras

`nvidia/segformer-b0-finetuned-ade-512-512` asigna una etiqueta a cada píxel
usando el dataset ADE20K (150 categorías: cielo, suelo, pared, persona…).

In [ ]:
from PIL import ImageOps

URL_CALLE = "https://upload.wikimedia.org/wikipedia/commons/thumb/9/9c/Golden_Gate_Bridge_from_Battery_Spencer.jpg/640px-Golden_Gate_Bridge_from_Battery_Spencer.jpg"

imagen_calle = cargar_imagen(URL_CALLE)
display(imagen_calle)

segmentador = pipeline(
    "image-segmentation",
    model="nvidia/segformer-b0-finetuned-ade-512-512"
)

mascaras = segmentador(imagen_calle)

print("Etiquetas detectadas:")
for m in mascaras:
    print(f"  {m['label']}")

In [ ]:
# Mostrar la máscara de cada etiqueta sobre la imagen original
for m in mascaras:
    base = imagen_calle.copy()
    mascara = ImageOps.invert(m["mask"])
    base.paste(mascara, mask=mascara)
    print(f"→ {m['label']}")
    display(base)

---
## 4 · Zero-shot con CLIP (categorías propias, sin entrenamiento)

CLIP aprende a alinear imágenes y descripciones de texto en el mismo espacio
vectorial. Puedes clasificar cualquier imagen con tus propias etiquetas en lenguaje
natural, sin haber visto ni un ejemplo de entrenamiento.

### 4a · Via pipeline (rápido)

In [ ]:
clasificador_clip = pipeline(
    "zero-shot-image-classification",
    model="openai/clip-vit-base-patch32"
)

URL_ANIMAL = (
    "https://upload.wikimedia.org/wikipedia/commons/thumb/"
    "3/3f/Walking_tiger_female.jpg/640px-Walking_tiger_female.jpg"
)
imagen_animal = cargar_imagen(URL_ANIMAL)
display(imagen_animal)

etiquetas = ["gato", "perro", "tigre", "oso", "coche"]
predicciones = clasificador_clip(imagen_animal, candidate_labels=etiquetas)

for p in predicciones:
    print(f"{p['label']:15s}  {p['score']:.3f}")

### 4b · API de bajo nivel (CLIPModel + CLIPProcessor)

Útil para entender cómo CLIP calcula las probabilidades internamente.

In [ ]:
from transformers import CLIPModel, CLIPProcessor

modelo_clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
procesador_clip = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
modelo_clip.eval()

etiquetas_en = ["cat", "dog", "tiger", "bear", "car"]
entradas_clip = procesador_clip(
    text=etiquetas_en,
    images=imagen_animal,
    return_tensors="pt",
    padding=True
)

with torch.no_grad():
    salidas_clip = modelo_clip(**entradas_clip)

# logits_per_image: puntuación de similitud imagen ↔ cada etiqueta
probs = salidas_clip.logits_per_image.softmax(dim=1)
idx_max = torch.argmax(probs, dim=1).item()

print("Probabilidades CLIP:")
for etiqueta, prob in zip(etiquetas_en, probs[0]):
    print(f"  {etiqueta:10s}  {prob.item():.3f}")
print(f"\nPredicción: {etiquetas_en[idx_max]}")

---
## 5 · Audio: transcripción con Whisper

Whisper (OpenAI) es un modelo encoder-decoder entrenado sobre 680.000 h de
audio. Detecta el idioma automáticamente y funciona bien en español.

In [ ]:
%pip install -q librosa soundfile

from transformers import pipeline

transcriptor = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-small"
)

# Opción A: archivo local
# print(transcriptor("mi_audio.wav"))

# Opción B: descarga un audio de muestra (CC0, LibriSpeech)
import urllib.request
urllib.request.urlretrieve(
    "https://www.voiptroubleshooter.com/open_speech/american/OSR_us_000_0010_8k.wav",
    "muestra.wav"
)

resultado_audio = transcriptor("muestra.wav")
print("Transcripción:", resultado_audio["text"])

---
## 6 · Multimodal: describir imágenes con image-to-text

Un modelo de *image captioning* toma una imagen y genera una descripción textual.
`Salesforce/blip-image-captioning-base` es ligero y funciona bien en Colab.

In [ ]:
descriptor = pipeline(
    "image-to-text",
    model="Salesforce/blip-image-captioning-base"
)

# Probamos con la imagen del tigre
descripcion = descriptor(imagen_animal)
print("Descripción:", descripcion[0]["generated_text"])

In [ ]:
# Visual Question Answering: pregunta sobre el contenido de una imagen
vqa = pipeline(
    "visual-question-answering",
    model="dandelin/vilt-b32-finetuned-vqa"
)

respuesta = vqa(
    image=imagen_animal,
    question="What animal is in the image?"
)
print("VQA:", respuesta[0]["answer"], f"(score {respuesta[0]['score']:.2f})")

---
## Práctica

**Encadena dos modalidades:**
1. Transcribe con Whisper el audio `muestra.wav` → obtienes texto.
2. Clasifica ese texto con el modelo de sentimiento de la sesión 38.
3. Muestra: texto transcrito + etiqueta de sentimiento + score.

In [ ]:
# TODO: encadena transcriptor (Whisper) + clasificador de sentimiento
# pista:
#   texto = transcriptor("muestra.wav")["text"]
#   resultado = clasificador_texto(texto)[0]
#   print(texto, resultado)

---
## Reto · Clasificador CLIP con tus propias categorías

Elige un tema que te interese (productos de una tienda, razas de perros,
tipos de paisaje…) y crea un clasificador zero-shot con CLIP:

1. Carga al menos **3 imágenes** (pueden ser URLs de Wikimedia o fotos propias).
2. Define **5–10 etiquetas** en lenguaje natural.
3. Para cada imagen, muestra la etiqueta más probable y su probabilidad.
4. ¿Hay casos donde CLIP se confunde? Explica por qué podría ocurrir.

In [ ]:
# TODO: clasificador CLIP con categorías y imágenes propias

mis_etiquetas = ["..."]  # define tus etiquetas
mis_imagenes = ["..."]   # URLs o rutas locales

for origen in mis_imagenes:
    img = cargar_imagen(origen)
    display(img)
    preds = clasificador_clip(img, candidate_labels=mis_etiquetas)
    print(f"  → {preds[0]['label']} ({preds[0]['score']:.3f})\n")

---
## Autoevaluación

| Pregunta | Respuesta |
|---|---|
| ¿Qué diferencia hay entre `image-classification` y `zero-shot-image-classification`? | |
| ¿Por qué DETR usa atención en lugar de propuestas de regiones clásicas? | |
| ¿Qué devuelve `post_process_object_detection`? ¿Qué es `threshold`? | |
| ¿Cómo sabe Whisper en qué idioma está el audio? | |
| ¿Qué aprende CLIP durante su preentrenamiento? | |